# 📊 Retail Sales Data Cleaning & Exploratory Data Analysis

## Project Overview

This notebook performs data cleaning, exploratory data analysis (EDA), feature engineering, and prepares the dataset for SQL, Power BI, and Streamlit dashboard development.


## 1. Import Libraries


In [ ]:
# Import libraries

import pandas as pd
import numpy as np
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

## 2. Load Dataset

In [ ]:
df = pd.read_csv("C:/Users/DELL/OneDrive/Desktop/Sales_data(EDA Exported).csv")

## 3. Data Exploration

In [ ]:
df.head()

In [ ]:
df.tail()

In [ ]:
df.shape

In [ ]:
df.columns

In [ ]:
df.info()

In [ ]:
df.describe()

## 4. Missing Value Analysis

In [ ]:
df.isnull().sum()

## 5. Duplicate Records

In [ ]:
df.duplicated().sum()

## Feature Engineering

New features were created from the Order Date column to improve time-series analysis and enable monthly, quarterly, and weekday-level business insights.

In [ ]:
df['order_date'] = pd.to_datetime(df['order_date'])
df['Year'] = df['order_date'].dt.year
df['Quarter'] = df['order_date'].dt.quarter
df['Month'] = df['order_date'].dt.month_name()
df['Weekday'] = df['order_date'].dt.day_name()
df['Week'] = df['order_date'].dt.isocalendar().week
df['Is_Weekend'] = df['Weekday'].isin(['Saturday','Sunday'])

In [ ]:
df.info()

In [ ]:
df.rename(columns={
    'order_number': 'Order_Number',
    'customer_name': 'Customer_Name',
    'product_name': 'Product_Name',
    'unit_price': 'Unit_Price',
    'state_name': 'State_Name',
    'total_cost': 'Total_Cost',
    'profit_margin_pct': 'Profit_Margin_Pct'
}, inplace=True)

In [ ]:
budget_df = df[df['budget'].notna()]

In [ ]:
budget_df.isnull().sum()

In [ ]:
budget_df.shape

## 7. Exploratory Data Analysis (EDA)

In [ ]:
df.to_csv("cleaned_sales_data.csv", index=False)

In [ ]:
# Makes graphs appear inside the notebook
%matplotlib inline

In [ ]:
df = pd.read_csv("cleaned_sales_data.csv")

In [ ]:
df['order_date'] = pd.to_datetime(df['order_date'])

In [ ]:
total_revenue = df['revenue'].sum()
print(f"Total Revenue: ${total_revenue:,.2f}")

In [ ]:
total_profit = df['profit'].sum()
print(f"Total Profit: ${total_profit:,.2f}")

In [ ]:
total_orders = df['Order_Number'].nunique()
print("Total Orders:", total_orders)

In [ ]:
total_customers = df['Customer_Name'].nunique()
print("Total Customers:", total_customers)

In [ ]:
aov = total_revenue / total_orders
print(f"Average Order Value: ${aov:,.2f}")

In [ ]:
top_products = (
    df.groupby('Product_Name')['revenue']
      .sum()
      .sort_values(ascending=False)
      .head(10)
)

top_products

In [ ]:
plt.figure(figsize=(10,5))

top_products.plot(kind='bar')

plt.title("Top 10 Products by Revenue")
plt.xlabel("Product")
plt.ylabel("Revenue")

plt.xticks(rotation=45)

plt.show()

In [ ]:
top_customers = (
    df.groupby('Customer_Name')['revenue']
      .sum()
      .sort_values(ascending=False)
      .head(10)
)

top_customers

In [ ]:
df['Month'] = df['order_date'].dt.to_period('M')

In [ ]:
monthly_sales = (
    df.groupby('Month')['revenue']
      .sum()
)

In [ ]:
plt.figure(figsize=(12,5))

monthly_sales.plot()

plt.title("Monthly Revenue Trend")

plt.ylabel("Revenue")

plt.show()

In [ ]:
region_sales = (
    df.groupby('us_region')['revenue']
      .sum()
      .sort_values(ascending=False)
)

region_sales

In [ ]:
region_sales.plot(kind='bar')
plt.title("Revenue by Region")
plt.show()

In [ ]:
customer_sales = (
    df.groupby('Customer_Name')['revenue']
      .sum()
      .reset_index()
)

customer_sales.head()

In [ ]:
customer_sales['Segment'] = pd.qcut(
    customer_sales['revenue'],
    q=3,
    labels=['Low Value','Regular','VIP']
)

In [ ]:
customer_sales.head()

In [ ]:
customer_rev = (
    df.groupby('Customer_Name')['revenue']
      .sum()
      .sort_values(ascending=False)
      .reset_index()
)

customer_rev['Cumulative %'] = (
    customer_rev['revenue'].cumsum() /
    customer_rev['revenue'].sum()
) * 100

In [ ]:
product_profit = (
    df.groupby('Product_Name')
      .agg(
          Revenue=('revenue','sum'),
          Profit=('profit','sum')
      )
      .sort_values(by='Profit', ascending=False)
)

product_profit.head(10)

In [ ]:
budget_df = df[df['budget'].notna()]

In [ ]:
budget_analysis = budget_df.groupby('Product_Name').agg(
    Budget=('budget','sum'),
    Revenue=('revenue','sum')
)

budget_analysis.head()

In [ ]:
budget_analysis['Budget Utilization %'] = (
    budget_analysis['Revenue'] /
    budget_analysis['Budget']
) * 100

In [ ]:
corr = df[['quantity', 'Unit_Price', 'revenue', 'Total_Cost', 'profit']].corr()

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(8,6))
sns.heatmap(corr, annot=True, cmap='coolwarm')
plt.title("Correlation Matrix")
plt.show()

## Save Cleaned Dataset

The cleaned dataset is exported and will be used for SQL analysis, Power BI dashboard, and Streamlit application.

In [ ]:
df.to_csv("sales_ready_for_powerbi.csv", index=False)